In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
print("TensorFlow version:", tf.__version__)
# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime
import shutil

TensorFlow version: 2.18.0


In [2]:
%reload_ext tensorboard
from tensorflow.keras import Model
from pathlib import Path
import pandas as pd

In [21]:
BATCH_SIZE = 64   # 64
BUFFER_SIZE = 512   # 512
LEARNING_RATE = 0.01 # 0.001  
EPOCHS = 4000 # 2000

In [22]:
input_dir = Path('./data/prepared')
logs_path = Path('./data/logs')
if logs_path.exists():
  shutil.rmtree(logs_path) # удаляем, если существует /logs
logs_path.mkdir(parents=True)

X_train_name = input_dir / 'X_train.csv'
y_train_name = input_dir / 'y_train.csv'
X_test_name = input_dir / 'X_test.csv'
y_test_name = input_dir / 'y_test.csv'

X_train = pd.read_csv(X_train_name)
y_train = pd.read_csv(y_train_name)
X_test = pd.read_csv(X_test_name)
y_test = pd.read_csv(y_test_name)

X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()

# print(X_train.isnull().sum())
# print(y_train.isnull().sum())
# print(X_train.dtypes)  # Типы данных в X_train
# print(y_train.dtypes)  # Типы данных в y_train


train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

In [23]:
@tf.keras.utils.register_keras_serializable() #  Декоратор позволяет сериализовать и десериализовать модель для сохранения и загрузки.
class SomeModel(Model):
    def __init__(self, neurons_cnt=64, **kwargs):
        super(SomeModel, self).__init__(**kwargs)
        self.neurons_cnt = neurons_cnt  # Сохраняем значение параметра для конфигурации
        self.d_in = Dense(17, activation='relu')
        self.d1 = Dense(neurons_cnt, activation='relu')
        self.d2 = Dense(neurons_cnt, activation='relu')
        self.d3 = Dense(neurons_cnt, activation='relu')
        self.d_out = Dense(1)

    def call(self, x):
        x = self.d_in(x)
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        return self.d_out(x)
         
    def build(self, input_shape): # надо явно определить для построения
        super(SomeModel, self).build(input_shape)
        
    def get_config(self): 
        # Возвращаем параметры модели, включая кастомные
        config = super(SomeModel, self).get_config()
        config.update({
            "neurons_cnt": self.neurons_cnt  # Добавляем кастомный параметр в конфигурацию
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Создаём экземпляр класса из конфигурации
        return cls(**config)

In [24]:
# Create an instance of the model
model = SomeModel(neurons_cnt=32) # 32
model.build(input_shape=(None, 17))  # 27

In [26]:
loss_object = tf.keras.losses.MeanSquaredError() # что? 
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.MeanAbsoluteError(name='test_mae')

In [27]:
@tf.function
def train_step(input_vector, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(input_vector, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

@tf.function
def test_step(input_vector, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(input_vector, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [28]:
from tensorflow import keras
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = logs_path / 'gradient_tape' / current_time / 'train'
train_log_dir.mkdir(exist_ok=True, parents=True)
test_log_dir = logs_path / 'gradient_tape' / current_time / 'test'
test_log_dir.mkdir(exist_ok=True, parents=True)
train_summary_writer = tf.summary.create_file_writer(str(train_log_dir))
test_summary_writer = tf.summary.create_file_writer(str(test_log_dir))

logdir = logs_path / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir.mkdir(exist_ok=True, parents=True)
fit_summary_writer = tf.summary.create_file_writer(str(logdir))

tf.summary.trace_on(graph=True, profiler=True, profiler_outdir=str(logdir))

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  for (x_train, y_train) in train_ds:

    with fit_summary_writer.as_default():
      train_step(x_train, y_train)


  with train_summary_writer.as_default():
    tf.summary.scalar('loss', train_loss.result(), step=epoch)
    tf.summary.scalar('accuracy', train_accuracy.result(), step=epoch)

  for (x_test, y_test) in test_ds:
    test_step(x_test, y_test)

  with test_summary_writer.as_default():
    tf.summary.scalar('loss', test_loss.result(), step=epoch)
    tf.summary.scalar('mae', test_accuracy.result(), step=epoch)
    for layer in model.layers:
        for weight in layer.weights:
            tf.summary.histogram(f"{layer.name}/{weight.name}", weight, step=epoch)
            
  template = 'Epoch {}, Loss: {}, Accuracy: {}, Test Loss: {}, Test MAE: {}'
  print (template.format(epoch+1,
                         train_loss.result(),
                         train_accuracy.result(),
                         test_loss.result(),
                         test_accuracy.result()))

  # Reset metrics every epoch
  train_loss.reset_state()
  test_loss.reset_state()
  train_accuracy.reset_state()
  test_accuracy.reset_state()

with fit_summary_writer.as_default():
  tf.summary.trace_export(
      name="my_func_trace",
      step=0,
      profiler_outdir=str(logdir)
  )

Epoch 1, Loss: 1544.7122802734375, Accuracy: 31.807178497314453, Test Loss: 337.8265380859375, Test MAE: 16.07292938232422
Epoch 2, Loss: 129.908935546875, Accuracy: 9.275827407836914, Test Loss: 51.46177291870117, Test MAE: 5.7150559425354
Epoch 3, Loss: 58.1557731628418, Accuracy: 5.9809675216674805, Test Loss: 46.436038970947266, Test MAE: 5.386915683746338
Epoch 4, Loss: 47.2245979309082, Accuracy: 5.33592414855957, Test Loss: 42.27082443237305, Test MAE: 5.014405250549316
Epoch 5, Loss: 40.541053771972656, Accuracy: 4.960453510284424, Test Loss: 35.476375579833984, Test MAE: 4.660408020019531
Epoch 6, Loss: 37.100467681884766, Accuracy: 4.7552103996276855, Test Loss: 35.902984619140625, Test MAE: 4.577246189117432
Epoch 7, Loss: 32.60550308227539, Accuracy: 4.439869403839111, Test Loss: 28.577133178710938, Test MAE: 3.987191677093506
Epoch 8, Loss: 28.603281021118164, Accuracy: 4.167916774749756, Test Loss: 26.854162216186523, Test MAE: 3.8401684761047363
Epoch 9, Loss: 26.0220432

In [20]:
%tensorboard --logdir ./data/logs/gradient_tape --port=8082

Reusing TensorBoard on port 8082 (pid 22624), started 0:06:02 ago. (Use '!kill 22624' to kill it.)